# Лабораторная 5. Задачи 6.2, 6.8, 6.11, 6.12

**6.2.** Найти первую квадратичную форму заданных поверхностей.

**6.8.** Найти ортогональные траектории семейства линий
$u+v=\mathrm{const}$, лежащих на сфере.

**6.11.** Для первой квадратичной формы
$ds^2=du^2+(u^2+a^2)dv^2$ найти периметр, углы и площадь указанных
криволинейных треугольников.

**6.12.** Для поверхности $r(u,v)=(u\sin v,u\cos v,v)$ найти площадь,
длины сторон и углы криволинейного треугольника
$0\le u\le\sinh v,\ 0\le v\le v_0$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["animation.embed_limit"] = 100


def finish_animation(anim, fig):
    html = HTML(anim.to_jshtml())
    plt.close(fig)
    return html


def set_axes_equal_3d(ax):
    x_limits = ax.get_xlim3d()
    y_limits = ax.get_ylim3d()
    z_limits = ax.get_zlim3d()
    x_range = abs(x_limits[1] - x_limits[0])
    y_range = abs(y_limits[1] - y_limits[0])
    z_range = abs(z_limits[1] - z_limits[0])
    radius = 0.5 * max([x_range, y_range, z_range])
    x_middle = np.mean(x_limits)
    y_middle = np.mean(y_limits)
    z_middle = np.mean(z_limits)
    ax.set_xlim3d([x_middle - radius, x_middle + radius])
    ax.set_ylim3d([y_middle - radius, y_middle + radius])
    ax.set_zlim3d([z_middle - radius, z_middle + radius])


def setup_3d(ax, xlim, ylim, zlim, title):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_zlim(*zlim)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title(title)

In [ ]:
# Динамическая иллюстрация к 6.2г: трубка вокруг винтовой линии.
a = 1.2
b = 0.35
radius = 0.28


def helix(t):
    return np.array([a * np.cos(t), a * np.sin(t), b * t])


def frame(t):
    c = np.sqrt(a**2 + b**2)
    T = np.array([-a * np.sin(t) / c, a * np.cos(t) / c, np.full_like(t, b / c)])
    N = np.array([-np.cos(t), -np.sin(t), np.zeros_like(t)])
    B = np.array([b * np.sin(t) / c, -b * np.cos(t) / c, np.full_like(t, a / c)])
    return T, N, B


t = np.linspace(0, 5 * np.pi, 160)
phi = np.linspace(0, 2 * np.pi, 28)
Tgrid, Phigrid = np.meshgrid(t, phi)
base = helix(Tgrid)
_, N, Bv = frame(Tgrid)
tube = base + radius * (N * np.cos(Phigrid) + Bv * np.sin(Phigrid))

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
setup_3d(ax, (-1.8, 1.8), (-1.8, 1.8), (-0.5, 6.0), "6.2: каналовая поверхность")
ax.plot_surface(tube[0], tube[1], tube[2], alpha=0.65, linewidth=0, color="steelblue")
center = helix(t)
ax.plot(center[0], center[1], center[2], color="navy", linewidth=2)
ring_line, = ax.plot([], [], [], color="crimson", linewidth=2)
set_axes_equal_3d(ax)


def animate(i):
    k = int(i / 100 * (len(t) - 1))
    ring = tube[:, :, k]
    ring_line.set_data(ring[0], ring[1])
    ring_line.set_3d_properties(ring[2])
    ax.view_init(elev=24, azim=35 + i)
    return ring_line,


anim = FuncAnimation(fig, animate, frames=101, interval=70)
finish_animation(anim, fig)

In [ ]:
R = 1.6


def sphere_uv(u, v):
    return np.array([R * np.cos(u) * np.cos(v), R * np.cos(u) * np.sin(v), R * np.sin(u)])


u = np.linspace(-1.1, 1.1, 300)
family = []
orth = []
for c0 in np.linspace(-2.0, 2.0, 9):
    family.append(sphere_uv(u, c0 - u))
    orth.append(sphere_uv(u, np.tan(u) + c0))

su = np.linspace(-np.pi / 2, np.pi / 2, 45)
sv = np.linspace(0, 2 * np.pi, 70)
SU, SV = np.meshgrid(su, sv)
S = sphere_uv(SU, SV)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
setup_3d(ax, (-1.8, 1.8), (-1.8, 1.8), (-1.8, 1.8), "6.8: семейство и ортогональные траектории")
ax.plot_surface(S[0], S[1], S[2], alpha=0.12, linewidth=0, color="steelblue")
for curve in family:
    ax.plot(curve[0], curve[1], curve[2], color="navy", linewidth=1)
for curve in orth:
    ax.plot(curve[0], curve[1], curve[2], color="crimson", linewidth=1)
set_axes_equal_3d(ax)


def animate(i):
    ax.view_init(elev=24, azim=35 + 2 * i)
    return []


anim = FuncAnimation(fig, animate, frames=120, interval=70)
finish_animation(anim, fig)

In [ ]:
# Динамика области из 6.12 на поверхности при росте v0.
v0_final = 1.4
frames = 90


def surf(u, v):
    return np.array([u * np.sin(v), u * np.cos(v), v])


fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")


def animate(i):
    ax.clear()
    v0 = 0.15 + (v0_final - 0.15) * i / (frames - 1)
    vv = np.linspace(0, v0, 70)
    ss = np.linspace(0, 1, 28)
    Sg, Vg = np.meshgrid(ss, vv)
    Ug = Sg * np.sinh(Vg)
    P = surf(Ug, Vg)
    setup_3d(ax, (-1.4, 1.4), (-0.2, 1.8), (-0.1, 1.6), f"6.12: область, v0={v0:.2f}")
    ax.plot_surface(P[0], P[1], P[2], alpha=0.70, linewidth=0, color="steelblue")
    for uu, vv_line, color in [
        (np.zeros_like(vv), vv, "crimson"),
        (np.linspace(0, np.sinh(v0), 80), np.full(80, v0), "darkorange"),
        (np.sinh(vv), vv, "purple"),
    ]:
        C = surf(uu, vv_line)
        ax.plot(C[0], C[1], C[2], color=color, linewidth=2.5)
    ax.view_init(elev=25, azim=35 + i)
    set_axes_equal_3d(ax)
    return []


anim = FuncAnimation(fig, animate, frames=frames, interval=80)
finish_animation(anim, fig)